<div align="center">

# ARITZIA-INSPIRED BUYER DASHBOARD
### Product Performance • Inventory • Pricing • Merchandising

</div>


**Purpose:** Demonstrate how an assistant buyer could use retail sales, inventory, pricing, markdown, seasonality, and product data to make better buying and merchandising decisions.

**Dataset:** [Retail Fashion Boutique Data – Sales Analytics 2025 (Kaggle)](https://www.kaggle.com/datasets/pratyushpuri/retail-fashion-boutique-data-sales-analytics-2025)

**Inspiration:** [Zara Sales Analysis dashboard](https://github.com/Mehdi-Benbiba/Zara-Sales-analysis/blob/main/Zara%20sales%20dashboard.pdf) — this project mirrors that single-page dashboard *layout* (KPI cards + charts + table together) while telling a *different* story (inventory/markdown/returns/ratings, since there's no sales-volume data here).


## Important Analytical Limitations

Being upfront about what this dataset can and cannot tell us:

**This dataset does NOT provide:**
- Actual sales volume
- Units sold
- Revenue
- Supplier information
- Supplier lead time
- Purchase orders
- Procurement cost
- Demand forecast

**Therefore, this dashboard does NOT claim to perform:**
- Sales forecasting
- Procurement optimization
- Demand forecasting
- Supplier optimization

**Instead, this dashboard is positioned as:**

> ### Assistant Buyer / Merchandising Decision Support

...focused specifically on **inventory, pricing, markdown, product assortment, customer ratings, and returns** — using only the fields that actually exist in the data.


### Guiding principles

1. Everything is calculated directly from the dataset — nothing is fabricated.
2. Stock Quantity is never treated as, or labeled as, sales volume.
3. All buyer recommendation rules in this notebook are an explicit **prototype / decision-support tool**, not a real retail buying methodology.
4. Return-related visuals describe *what* is happening, not *why*, since the dataset doesn't support causal claims beyond the stated `return_reason` categories.
5. The Stock Quantity vs. Markdown % scatter plot is the analytical centerpiece of this dashboard.

**Out of scope (by design):** No machine learning, demand forecasting, supplier management, purchase orders, SQL database, login system, AI chatbot, external APIs, complex optimization, or fabricated columns (e.g. fake sales or supplier data).


---


## 0. Setup

Run this cell first. If you're in Google Colab and haven't uploaded the CSV yet, uncomment the `files.upload()` block to pick it from your computer. If the CSV is already sitting next to this notebook (e.g. running locally), just leave it as-is.


In [ ]:
# If running in Google Colab and you need to upload the CSV, uncomment these two lines:
# from google.colab import files
# uploaded = files.upload()  # select fashion_boutique_dataset.csv when prompted

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

DATA_PATH = 'fashion_boutique_dataset.csv'  # update path if needed
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")


## 1. Load & Inspect the Data

Confirming exact column names, types, and data quality before building any rules or charts — nothing downstream should assume a column exists that isn't actually here.


In [ ]:
df.dtypes


In [ ]:
df.head(10)


In [ ]:
# Null counts per column
df.isnull().sum().to_frame('null_count').assign(
    null_pct=lambda x: (x['null_count'] / len(df) * 100).round(1)
)


In [ ]:
# Category-type field values, to confirm exact labels used downstream
for col in ['category', 'brand', 'season', 'size']:
    print(f"{col}: {sorted(df[col].dropna().unique())}")


**Data quality notes (confirmed above):**

- `size` is null for ~23% of rows (likely one-size-fits-all items like some accessories) — left as-is, not imputed.
- `customer_rating` is null for ~17% of rows. Per project decision: missing ratings are **excluded** from the buyer-recommendation rule logic (they don't count as "good" or "poor") and are **excluded** from rating-based visualizations, but the underlying rows are still included normally in KPIs and the product table.
- `is_returned` is boolean; `return_reason` is only populated when `is_returned` is `True`.
- `product_id` is unique per row (no duplicates) — each row represents one product/SKU record, not an aggregated product line.
- There is no sales-volume or units-sold column anywhere in this dataset. `stock_quantity` is inventory on hand, not units sold.


## 2. Data Preparation

Before building KPIs, the recommendation engine, or charts, we derive a few reusable fields:

- `purchase_date` parsed to an actual datetime.
- **Stock tier** (Low / Moderate / High) — split into terciles (roughly equal-sized thirds) directly from the data's own distribution.
- **Markdown tier** (Low / Moderate / High) — `0%` markdown is treated as "Low/none" (this is true for ~60% of rows), anything above the 75th percentile of markdown is "High", and everything else is "Moderate".
- **Rating tier** (Good / Neutral / Poor / Unknown) — `Good` ≥ 4.0, `Poor` ≤ 2.0, `Unknown` = missing rating (never treated as good or poor).

These tiers are used consistently across the Buyer Recommendation rules and the Buyer Attention alerts further down, so the whole dashboard applies the same definitions of "low stock" / "high markdown" / "good rating" everywhere.


In [ ]:
df['purchase_date'] = pd.to_datetime(df['purchase_date'], errors='coerce')

# --- Stock tier: terciles of stock_quantity ---
df['stock_tier'] = pd.qcut(df['stock_quantity'], 3, labels=['Low', 'Moderate', 'High'])

# --- Markdown tier: 0% = Low/none, >75th percentile = High, else Moderate ---
markdown_high_cutoff = df['markdown_percentage'].quantile(0.75)

def _markdown_tier(x, cutoff=markdown_high_cutoff):
    if x == 0:
        return 'Low'
    elif x <= cutoff:
        return 'Moderate'
    else:
        return 'High'

df['markdown_tier'] = df['markdown_percentage'].apply(_markdown_tier)

# --- Rating tier: Good >=4.0, Poor <=2.0, Unknown = missing, else Neutral ---
def _rating_tier(x):
    if pd.isna(x):
        return 'Unknown'
    elif x >= 4.0:
        return 'Good'
    elif x <= 2.0:
        return 'Poor'
    else:
        return 'Neutral'

df['rating_tier'] = df['customer_rating'].apply(_rating_tier)

print(f"Markdown 'High' cutoff (75th percentile): {markdown_high_cutoff:.1f}%")
print()
print("Stock tier counts:\n", df['stock_tier'].value_counts(), sep='')
print()
print("Markdown tier counts:\n", df['markdown_tier'].value_counts(), sep='')
print()
print("Rating tier counts:\n", df['rating_tier'].value_counts(), sep='')


---


## Executive Dashboard

One combined view — styled KPI cards followed by all six required charts and the Top 10 table in a single grid, mirroring the Zara reference's single-page layout.


In [ ]:
total_inventory = int(df['stock_quantity'].sum())
total_products = int(df['product_id'].nunique())
avg_price = df['current_price'].mean()
avg_markdown = df['markdown_percentage'].mean()
return_rate = df['is_returned'].mean() * 100

kpi_labels = ['Total Inventory', 'Products', 'Avg Price', 'Avg Markdown', 'Return Rate']
kpi_display = [f"{total_inventory:,}", f"{total_products:,}", f"${avg_price:,.2f}",
               f"{avg_markdown:.1f}%", f"{return_rate:.1f}%"]
kpi_colors = ['#2C3E50', '#2C3E50', '#2C3E50', '#C0392B', '#C0392B']

cards_html = ''.join(f'''
    <div style="flex:1; background:#ffffff; border-radius:10px;
                box-shadow:0 2px 10px rgba(0,0,0,0.15); padding:22px 12px;
                margin:0 8px; text-align:center; border-top:4px solid {color};">
        <div style="font-size:12px; font-weight:700; letter-spacing:1.2px; color:#7f8c8d;
                    text-transform:uppercase; margin-bottom:10px;">{label}</div>
        <div style="font-size:30px; font-weight:700; color:{color};">{value}</div>
    </div>
''' for label, value, color in zip(kpi_labels, kpi_display, kpi_colors))

display(HTML(f'''
<div style="display:flex; justify-content:space-between; padding:10px 4px 20px 4px;
            font-family:Arial, Helvetica, sans-serif; background:#f4f6f7; border-radius:12px;">
    {cards_html}
</div>
'''))


In [ ]:
fig_dashboard = make_subplots(
    rows=3, cols=2,
    specs=[
        [{'type': 'xy'},     {'type': 'xy'}],
        [{'type': 'domain'}, {'type': 'xy'}],
        [{'type': 'xy'},     {'type': 'table'}],
    ],
    subplot_titles=(
        'Inventory by Category', 'Stock Quantity vs. Markdown %',
        'Inventory Distribution by Season', 'Average Markdown by Season',
        'Return Rate by Category', 'Top 10 Products by Inventory'
    ),
    vertical_spacing=0.08,
    horizontal_spacing=0.12,
    row_heights=[0.34, 0.33, 0.33],
)

# --- (1,1) Inventory by Category — horizontal bar ---
# Business question: Where are we carrying the most inventory?
inv_by_category = df.groupby('category', as_index=False)['stock_quantity'].sum() \
    .sort_values('stock_quantity')

fig_dashboard.add_trace(go.Bar(
    x=inv_by_category['stock_quantity'], y=inv_by_category['category'],
    orientation='h', marker_color='#2C3E50',
    text=inv_by_category['stock_quantity'], texttemplate='%{text:,}', textposition='outside',
    showlegend=False
), row=1, col=1)

# --- (1,2) Stock Quantity vs. Markdown % — scatter (analytical centerpiece) ---
# Business question: relationship between inventory levels and markdowns.
# Quadrants (dashed lines = dataset medians): high stock+high markdown = potential
# overstock/review; high stock+low markdown = monitor; low stock+low markdown =
# healthy; low stock+high markdown = investigate. Prototype lens only.
median_stock = df['stock_quantity'].median()
median_markdown = df['markdown_percentage'].median()

scatter_fig = px.scatter(
    df, x='markdown_percentage', y='stock_quantity', color='category', opacity=0.6,
    hover_data=['product_id', 'brand', 'current_price']
)
for trace in scatter_fig.data:
    trace.showlegend = False
    fig_dashboard.add_trace(trace, row=1, col=2)

fig_dashboard.add_vline(x=median_markdown, line_dash='dash', line_color='gray', row=1, col=2)
fig_dashboard.add_hline(y=median_stock, line_dash='dash', line_color='gray', row=1, col=2)

# --- (2,1) Inventory Distribution by Season — donut ---
# Business question: which seasons represent the largest share of our inventory?
inv_by_season = df.groupby('season', as_index=False)['stock_quantity'].sum()

fig_dashboard.add_trace(go.Pie(
    labels=inv_by_season['season'], values=inv_by_season['stock_quantity'],
    hole=0.45, textinfo='label+percent', showlegend=False
), row=2, col=1)

# --- (2,2) Average Markdown by Season — bar ---
# Business question: which seasons are relying most heavily on markdowns?
markdown_by_season = df.groupby('season', as_index=False)['markdown_percentage'].mean() \
    .sort_values('markdown_percentage', ascending=False)

fig_dashboard.add_trace(go.Bar(
    x=markdown_by_season['season'], y=markdown_by_season['markdown_percentage'],
    marker_color='#C0392B', text=markdown_by_season['markdown_percentage'],
    texttemplate='%{text:.1f}%', textposition='outside', showlegend=False
), row=2, col=2)

# --- (3,1) Return Rate by Category — bar ---
# Business question: are certain categories experiencing more returns?
# Describes *what* is happening, not *why*.
return_rate_by_category = df.groupby('category', as_index=False)['is_returned'].mean()
return_rate_by_category['return_rate_pct'] = return_rate_by_category['is_returned'] * 100
return_rate_by_category = return_rate_by_category.sort_values('return_rate_pct', ascending=False)

fig_dashboard.add_trace(go.Bar(
    x=return_rate_by_category['category'], y=return_rate_by_category['return_rate_pct'],
    marker_color='#8E44AD', text=return_rate_by_category['return_rate_pct'],
    texttemplate='%{text:.1f}%', textposition='outside', showlegend=False
), row=3, col=1)

# --- (3,2) Top 10 Products by Inventory — table ---
# NOTE: this dataset has no sales-volume data, so these are never labeled "best
# sellers" (unlike the Zara reference) — they are simply the highest on-hand stock.
top10_cols = ['product_id', 'category', 'brand', 'stock_quantity', 'current_price', 'markdown_percentage']
top10_inventory = df.nlargest(10, 'stock_quantity')[top10_cols]

fig_dashboard.add_trace(go.Table(
    header=dict(
        values=['Product', 'Category', 'Brand', 'Stock Qty', 'Price', 'Markdown %'],
        fill_color='#2C3E50', font=dict(color='white', size=11), align='left'
    ),
    cells=dict(
        values=[top10_inventory[c] for c in top10_cols],
        align='left',
        format=[None, None, None, None, '$.2f', '.1f']
    )
), row=3, col=2)

# --- Axis labels ---
fig_dashboard.update_xaxes(title_text='Total Stock Quantity', row=1, col=1)
fig_dashboard.update_yaxes(title_text='Category', row=1, col=1)
fig_dashboard.update_xaxes(title_text='Markdown %', row=1, col=2)
fig_dashboard.update_yaxes(title_text='Stock Quantity', row=1, col=2)
fig_dashboard.update_xaxes(title_text='Season', row=2, col=2)
fig_dashboard.update_yaxes(title_text='Avg Markdown %', row=2, col=2)
fig_dashboard.update_xaxes(title_text='Category', row=3, col=1)
fig_dashboard.update_yaxes(title_text='Return Rate (%)', row=3, col=1)

fig_dashboard.update_layout(
    height=1150,
    showlegend=False,
    margin=dict(l=10, r=10, t=50, b=10),
    paper_bgcolor='white',
)
fig_dashboard.show()


_Quadrant guide for the Stock Quantity vs. Markdown % chart above:_
- **High stock + high markdown** → potential overstock / review
- **High stock + low markdown** → monitor inventory
- **Low stock + low markdown** → potentially healthy product
- **Low stock + high markdown** → investigate

*These quadrants are a prototype lens for exploration, not a claim about any retailer's actual buying rules.*


---


## Buyer Recommendations (Rule-Based Prototype)

> ⚠️ **This is an illustrative, rule-based prototype only** — not Aritzia's (or any retailer's) actual buying methodology. It is a decision-support demo built purely from the tiers defined in §2.

| Flag | Criteria |
|---|---|
| 🔥 **BUY / PRIORITIZE** | Low stock tier **and** Low markdown tier **and** Good rating tier |
| 🛑 **HOLD / REVIEW** | High stock tier **and** High markdown tier **and** (Poor rating tier **or** returned) |
| 👀 **MONITOR** | Everything else (the default / moderate case) |

Rows with an `Unknown` rating tier (missing `customer_rating`) can never qualify for `BUY` (which requires a confirmed Good rating), but can still be flagged `HOLD` if stock/markdown are both high and the item was returned.


In [ ]:
def buyer_recommendation(row):
    if row['stock_tier'] == 'Low' and row['markdown_tier'] == 'Low' and row['rating_tier'] == 'Good':
        return '🔥 BUY / PRIORITIZE'
    if row['stock_tier'] == 'High' and row['markdown_tier'] == 'High' and (
        row['rating_tier'] == 'Poor' or row['is_returned'] == True
    ):
        return '🛑 HOLD / REVIEW'
    return '👀 MONITOR'

df['buyer_recommendation'] = df.apply(buyer_recommendation, axis=1)

rec_counts = df['buyer_recommendation'].value_counts()
rec_pct = df['buyer_recommendation'].value_counts(normalize=True).mul(100).round(1)

pd.DataFrame({'Count': rec_counts, 'Share of Products': rec_pct.astype(str) + '%'})


## Product Performance Table

Per-product view including the rule-based Buyer Recommendation above.


In [ ]:
product_performance = df[[
    'product_id', 'category', 'brand', 'current_price', 'stock_quantity',
    'markdown_percentage', 'customer_rating', 'buyer_recommendation'
]].rename(columns={
    'product_id': 'Product',
    'category': 'Category',
    'brand': 'Brand',
    'current_price': 'Current Price',
    'stock_quantity': 'Stock Quantity',
    'markdown_percentage': 'Markdown %',
    'customer_rating': 'Rating',
    'buyer_recommendation': 'Buyer Recommendation'
})

print(f"Full table: {len(product_performance):,} products")
product_performance.head(20)


_The full `product_performance` DataFrame contains all 2,176 rows and can be exported (e.g. `product_performance.to_csv('product_performance.csv', index=False)`) — only the first 20 rows are shown above to keep the notebook readable._

---


## Buyer Attention

Rule-based, **not AI-generated** — every alert below is calculated directly from the dataset using the same tiers defined earlier. Language is deliberately hedged: these are prompts to review, not purchasing decisions.


In [ ]:
alerts = []

# --- Overstock Review: high stock tier AND high markdown tier ---
overstock = df[(df['stock_tier'] == 'High') & (df['markdown_tier'] == 'High')]
alerts.append({
    'Alert': 'Overstock Review',
    'Trigger': 'High stock quantity + high markdown %',
    'Products flagged': len(overstock),
    'Recommendation': 'Review for potential overstock.'
})

# --- Markdown Review: categories whose average markdown % exceeds the overall average ---
overall_avg_markdown = df['markdown_percentage'].mean()
cat_markdown = df.groupby('category')['markdown_percentage'].mean().sort_values(ascending=False)
flagged_markdown_categories = cat_markdown[cat_markdown > overall_avg_markdown]
alerts.append({
    'Alert': 'Markdown Review',
    'Trigger': f'Category avg markdown % above overall average ({overall_avg_markdown:.1f}%)',
    'Products flagged': ', '.join(f"{cat} ({val:.1f}%)" for cat, val in flagged_markdown_categories.items()),
    'Recommendation': 'Review markdown strategy.'
})

# --- Return Review: categories whose return rate exceeds the overall return rate ---
overall_return_rate = df['is_returned'].mean() * 100
cat_return = df.groupby('category')['is_returned'].mean().mul(100).sort_values(ascending=False)
flagged_return_categories = cat_return[cat_return > overall_return_rate]
alerts.append({
    'Alert': 'Return Review',
    'Trigger': f'Category return rate above overall rate ({overall_return_rate:.1f}%)',
    'Products flagged': ', '.join(f"{cat} ({val:.1f}%)" for cat, val in flagged_return_categories.items()),
    'Recommendation': 'Review product/return patterns.'
})

# --- Low Inventory: products in the Low stock tier ---
low_inventory = df[df['stock_tier'] == 'Low']
alerts.append({
    'Alert': 'Low Inventory',
    'Trigger': 'Stock quantity in bottom tercile',
    'Products flagged': len(low_inventory),
    'Recommendation': 'Monitor availability.'
})

alerts_df = pd.DataFrame(alerts)
alerts_df


---


## Optional: Price vs. Customer Rating

Rows with a missing `customer_rating` are excluded from this chart only (they carry no rating information to plot), consistent with the earlier decision to keep rating-based visuals limited to rated products.

- **High price + high rating** → potential premium performer
- **Low price + high rating** → potential value opportunity
- **High price + low rating** → review
- **Low price + low rating** → potentially weak product


In [ ]:
rated_df = df.dropna(subset=['customer_rating'])
median_price = rated_df['current_price'].median()
median_rating = rated_df['customer_rating'].median()

fig7 = px.scatter(
    rated_df,
    x='current_price', y='customer_rating',
    color='category',
    hover_data=['product_id', 'brand'],
    title='Price vs. Customer Rating',
    labels={'current_price': 'Current Price ($)', 'customer_rating': 'Customer Rating'},
    opacity=0.6,
)
fig7.add_vline(x=median_price, line_dash='dash', line_color='gray')
fig7.add_hline(y=median_rating, line_dash='dash', line_color='gray')
fig7.update_layout(height=520, margin=dict(l=10, r=10, t=60, b=10))
fig7.show()

print(f"Note: {df['customer_rating'].isnull().sum()} of {len(df)} rows excluded from this chart (missing rating).")


---


## Summary

This notebook is positioned as **Assistant Buyer / Merchandising Decision Support** — not sales forecasting, procurement optimization, demand forecasting, or supplier optimization, none of which this data can support.

It builds, in order:

1. **Executive Dashboard** — styled KPI cards + all 6 required charts + Top 10 table, combined in a single dashboard grid
2. **Buyer Recommendations** — rule-based prototype flag per product
3. **Product Performance Table** — full per-product view with recommendations
4. **Buyer Attention Panel** — 4 rule-based alerts, calculated directly from the data
5. **Optional** — Price vs. Customer Rating chart

All numbers are calculated directly from `fashion_boutique_dataset.csv` — nothing is fabricated, and stock quantity is never presented or implied as sales volume. The buyer-recommendation and alert rules are an explicit prototype for decision-support demonstration purposes only.
